# Train: QLoRA SFT on MetaMathQA

Colab front-end for `src/train.py`. Run one ablation config per session.

**Recommended runtime:** A100 (paid). T4 works for the smaller configs but is slow.

1. Set runtime to GPU (T4 free, A100 paid).
2. Run all cells.
3. Adapter saved to Google Drive automatically. ~5 MB per run.

## 1. Mount Drive + Clone + Install

Mount Drive first so outputs persist without manual copy steps.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_RUNS = '/content/drive/MyDrive/finetune-gsm8k-runs'
os.makedirs(DRIVE_RUNS, exist_ok=True)

if not os.path.isdir('/content/finetune-gsm8k'):
    !git clone https://github.com/YuZh98/finetune-gsm8k.git /content/finetune-gsm8k
os.chdir('/content/finetune-gsm8k')
!pip install -q -r requirements.txt

## 2. Sanity check: one tokenized example

Always print one full example before training. Catches chat-template bugs.

In [ ]:
from src.data import load_metamath
from src.utils import load_tokenizer

tok = load_tokenizer(padding_side='right')
ds = load_metamath(8)
print(tok.apply_chat_template(ds[0]['messages'], tokenize=False))

## 3. Pick the ablation row

Change `RUN_INDEX` to select which config to train. See `src/config.py` for the full matrix.

| Index | Run ID | What it tests |
|-------|--------|---------------|
| 1 | run1_r8 | Smallest rank |
| 2 | run2_r16 | Center config |
| 3 | run3_r64 | Largest rank |
| 4 | run4_mlp | All linear layers |
| 5 | run5_lr_low | LR = 5e-5 |
| 6 | run6_lr_high | LR = 1e-3 |
| 7 | run7_data5k | 5k data only |

In [ ]:
from src.config import (
    ABLATION_MATRIX,
    DEFAULT_TARGET_MODULES_ATTN,
    DEFAULT_TARGET_MODULES_ALL,
)

# ──── CHANGE THIS ────
RUN_INDEX = 2
# ─────────────────────

cfg = ABLATION_MATRIX[RUN_INDEX]
RUN_ID = cfg.run_id
RANK = cfg.rank
ALPHA = cfg.alpha
LR = cfg.lr
DATA = cfg.data_size

if cfg.target_modules == DEFAULT_TARGET_MODULES_ATTN:
    TARGET = 'attn'
elif cfg.target_modules == DEFAULT_TARGET_MODULES_ALL:
    TARGET = 'all'
else:
    raise ValueError(f'Unknown target_modules for {RUN_ID}: {cfg.target_modules}')

EPOCHS = 1
BATCH_SIZE = 4
GRAD_ACCUM = 4

print(f'{RUN_ID}: rank={RANK} alpha={ALPHA} target={TARGET} lr={LR} data={DATA}')

## 4. Train

Outputs directly to Drive — no manual save needed after training finishes.

In [ ]:
from src.train import train_one_run

OUTPUT_DIR = f'{DRIVE_RUNS}/{RUN_ID}'

train_one_run(
    rank=RANK,
    alpha=ALPHA,
    target=TARGET,
    lr=LR,
    data_size=DATA,
    output_dir=OUTPUT_DIR,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    grad_accum=GRAD_ACCUM,
)
print(f'\n✓ Adapter saved to Drive: {OUTPUT_DIR}/adapter')